In [0]:
%run ./00_config

In [0]:
#parameter,logging and a stop function
import json, os
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("bronze_path", "")
bronze_path = dbutils.widgets.get("bronze_path").strip()
source_file = os.path.basename(bronze_path)

def log_run(status, detail):
    (spark.createDataFrame([(source_file, status, json.dumps(detail))],
                           "file_name string, status string, detail string")
       .select(F.current_timestamp().alias("run_at"), F.lit("bronze_to_silver").alias("stage"),
               "file_name", "status", "detail")
       .write.mode("append").saveAsTable("nhsae.ops.run_log"))

def stop(reason, **extra):
    detail = {"status": "failed", "file": source_file, "reason": reason, **extra}
    log_run("failed", detail)
    raise RuntimeError(json.dumps(detail))

print(bronze_path)
if not bronze_path:
    stop("no bronze_path_given")
if not bronze_path.endswith(".csv"):
    stop("bronze_path_must_be_a_csv_file", path=bronze_path)    
    

In [0]:
#read, split total, rename, cast
df = spark.read.format("csv").option("header", True).load(bronze_path)
total_df = df.filter(F.col("Org Code") == "TOTAL")
org_df   = df.filter((F.col("Org Code") != "TOTAL") | F.col("Org Code").isNull())
if total_df.count() != 1:
    stop("total_row_missing_or_duplicated", total_rows=total_df.count())

renamed = org_df.select([F.col(src).alias(dst) for src, dst in COLUMN_MAP.items()])
typed = renamed.select(
    *[F.trim(F.col(c)).alias(c) for c in TEXT_COLS],
    *[F.expr(f"try_cast({c} AS INT)").alias(c) for c in NUMERIC_COLS],
    *[F.col(c).alias(f"{c}_raw") for c in NUMERIC_COLS],
).withColumn("period_date", F.expr(
    "try_to_date(initcap(regexp_extract(period, '([A-Za-z]+-[0-9]{4})$', 1)), 'MMMM-yyyy')"))

periods = [r[0] for r in typed.select("period_date").distinct().collect()]
if len(periods) != 1 or periods[0] is None:
    stop("bad_period", found=[str(p) for p in periods])
period = periods[0]
print("Org rows:", typed.count(), "| Period:", period)

In [0]:
#DQ rules an 5% threshold
PAIRS = [("att_t1_unbooked","o4_t1_unbooked"), ("att_t2_unbooked","o4_t2_unbooked"),
         ("att_oth_unbooked","o4_oth_unbooked"), ("att_t1_booked","o4_t1_booked"),
         ("att_t2_booked","o4_t2_booked"), ("att_oth_booked","o4_oth_booked")]
w = Window.partitionBy("org_code", "period_date")
checked = typed.withColumn("dup_count", F.count("*").over(w))

RULES = {
    "missing_org_code": F.col("org_code").isNull() | (F.col("org_code") == ""),
    "failed_type_conversion": reduce(lambda a, b: a | b,
        [F.col(c).isNull() & (F.trim(F.col(f"{c}_raw")) != "") for c in NUMERIC_COLS]),
    "negative_value": F.least(*[F.col(c) for c in NUMERIC_COLS]) < 0,
    "over_4hr_exceeds_attendances": reduce(lambda a, b: a | b, [F.col(o) > F.col(a) for a, o in PAIRS]),
    "duplicate_org_period": F.col("dup_count") > 1,
}
checked = checked.withColumn("dq_reason",
    F.concat_ws("; ", *[F.when(cond, F.lit(name)) for name, cond in RULES.items()]))

valid   = checked.filter(F.col("dq_reason") == "").drop("dup_count")
invalid = checked.filter(F.col("dq_reason") != "")
n_valid, n_invalid = valid.count(), invalid.count()
pct_invalid = 100 * n_invalid / max(n_valid + n_invalid, 1)
print(f"Valid {n_valid} | Invalid {n_invalid} | {pct_invalid:.1f}%")

if n_invalid > 0:
    (invalid.select(F.lit(source_file).alias("source_file"), "period_date", "org_code", "org_name",
                    "dq_reason",
                    F.to_json(F.struct(*[f"{c}_raw" for c in NUMERIC_COLS])).alias("raw_values"),
                    F.current_timestamp().alias("rejected_at"))
       .write.mode("append").saveAsTable("nhsae.ops.dq_rejects"))

if pct_invalid > 5:
    stop("too_many_invalid_rows", invalid=n_invalid, pct=round(pct_invalid, 2))

In [0]:
#reconcile the total rows
total_vals = total_df.select([F.expr(f"try_cast(`{src}` AS INT)").alias(dst)
                              for src, dst in COLUMN_MAP.items() if dst in NUMERIC_COLS]).first().asDict()
org_sums = typed.agg(*[F.sum(c).alias(c) for c in NUMERIC_COLS]).first().asDict()
mismatches = {c: [org_sums[c], total_vals[c]] for c in NUMERIC_COLS if org_sums[c] != total_vals[c]}
if mismatches:
    stop("reconciliation_failed", mismatches=mismatches)
print("Reconciled: organisation rows sum exactly to the TOTAL row")

In [0]:
#silver ae by type
#STACK(6,..) unpivots. it turns each wide row into 6 long rows, taking 4 values
#at a time:type,route,attendances, and over_4hr, in a wide-->long design
by_type = valid.select(
    "period_date", "org_code",
    F.expr("""stack(6,
        'Type 1', 'unbooked', att_t1_unbooked, o4_t1_unbooked,
        'Type 2', 'unbooked', att_t2_unbooked, o4_t2_unbooked,
        'Other',  'unbooked', att_oth_unbooked, o4_oth_unbooked,
        'Type 1', 'booked',   att_t1_booked,   o4_t1_booked,
        'Type 2', 'booked',   att_t2_booked,   o4_t2_booked,
        'Other',  'booked',   att_oth_booked,  o4_oth_booked
    ) AS (dept_type, route, attendances, over_4hr)""")
)
print(by_type.count())  # valid orgs x 6
#display(by_type.orderBy("org_code", "route", "dept_type"))

#silver ae_org monthly
''''In spark, number+ null=null,so if one of the three admission column is ever empty,
the total becomes null rather than ignoring the gap. A data qua;ity check'''''

org_monthly = valid.select(
    "period_date", "org_code", "org_name", "parent_org",
    "dta_4_12hr", "dta_12hr_plus",
    "adm_ae_t1", "adm_ae_t2", "adm_ae_oth", "adm_other",
).withColumn(
    "emergency_admissions_ae",
    F.col("adm_ae_t1") + F.col("adm_ae_t2") + F.col("adm_ae_oth")
)
print(org_monthly.count())
#display(org_monthly)


In [0]:
#writing down the month (idempotent)


       
  

def write_month(df, table):
    (df.withColumn("source_file", F.lit(source_file))
       .withColumn("ingested_at", F.current_timestamp())
       .write.format("delta")
       .mode("overwrite")
       .option("replaceWhere", f"period_date = '{period}'")
       .saveAsTable(table))

write_month(by_type, "nhsae.silver.ae_by_type")
write_month(org_monthly, "nhsae.silver.ae_org_monthly")
print(f"Written period {period} from {source_file}")
result = {"status": "silver", "file": source_file, "period": str(period),
          "valid_rows": n_valid, "invalid_rows": n_invalid}
log_run("silver", result)
dbutils.notebook.exit(json.dumps(result))

In [0]:
display(spark.sql("""
  SELECT 'by_type' AS tbl, period_date, source_file, COUNT(*) AS rows
  FROM nhsae.silver.ae_by_type GROUP BY period_date, source_file
  UNION ALL
  SELECT 'org_monthly', period_date, source_file, COUNT(*)
  FROM nhsae.silver.ae_org_monthly GROUP BY period_date, source_file
  ORDER BY tbl, period_date
"""))